In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!ls "/content/drive/MyDrive/EdTech Project/Code/data"
!ls "/content/drive/MyDrive/EdTech Project/Code/data/feature"
!ls "/content/drive/MyDrive/EdTech Project/Code/data/OULAD"

data_transform	feature  OULAD	README.md  synthetic
feature_engineered.csv	feature_table.csv
assessments.csv  studentAssessment.csv	studentRegistration.csv  vle.csv
courses.csv	 studentInfo.csv	studentVle.csv


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import missingno as msno
from plotnine import *
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [5]:
feature_table = pd.read_csv("/content/drive/MyDrive/EdTech Project/Code/data/feature/feature_table.csv")
feature_table

,code_module,code_presentation,module_presentation_length,historical_student_count,pct_male,pct_higher_ed,total_assignments,is_exam_included,total_learning_materials,day_offset,marketing_spend,marketing_campaign_flag,holiday_flag,semester_start_flag,semester_end_flag,enrollment_num_daily,day_of_week,is_weekend
0,AAA,2013J,268,383,0.610966,0.428198,6,1,413,0,25.72,0,0,1,0,55,0,0
1,AAA,2013J,268,383,0.610966,0.428198,6,1,413,1,25.72,0,0,1,0,57,1,0
2,AAA,2013J,268,383,0.610966,0.428198,6,1,413,2,25.72,0,0,1,0,62,2,0
3,AAA,2013J,268,383,0.610966,0.428198,6,1,413,3,25.72,0,0,1,0,51,3,0
4,AAA,2013J,268,383,0.610966,0.428198,6,1,413,4,25.72,0,0,1,0,87,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5495,GGG,2014B,241,833,0.200480,0.216086,10,1,367,245,36.55,0,0,0,1,6,0,0
5496,GGG,2014B,241,833,0.200480,0.216086,10,1,367,246,36.55,0,0,0,1,6,1,0
5497,GGG,2014B,241,833,0.200480,0.216086,10,1,367,247,36.55,0,0,0,1,19,2,0
5498,GGG,2014B,241,833,0.200480,0.216086,10,1,367,248,839.26,1,0,0,1,91,3,0


In [6]:
assessments = pd.read_csv("/content/drive/MyDrive/EdTech Project/Code/data/OULAD/assessments.csv")
courses = pd.read_csv("/content/drive/MyDrive/EdTech Project/Code/data/OULAD/courses.csv")
studentAssessment = pd.read_csv("/content/drive/MyDrive/EdTech Project/Code/data/OULAD/studentAssessment.csv")
studentInfo = pd.read_csv("/content/drive/MyDrive/EdTech Project/Code/data/OULAD/studentInfo.csv")
studentRegistration = pd.read_csv("/content/drive/MyDrive/EdTech Project/Code/data/OULAD/studentRegistration.csv")
studentVle = pd.read_csv("/content/drive/MyDrive/EdTech Project/Code/data/OULAD/studentVle.csv")
vle = pd.read_csv("/content/drive/MyDrive/EdTech Project/Code/data/OULAD/vle.csv")

In [6]:
feature_table

,code_module,code_presentation,module_presentation_length,historical_student_count,pct_male,pct_higher_ed,total_assignments,is_exam_included,total_learning_materials,day_offset,marketing_spend,marketing_campaign_flag,holiday_flag,semester_start_flag,semester_end_flag,enrollment_num_daily,day_of_week,is_weekend
0,AAA,2013J,268,383,0.610966,0.428198,6,1,413,0,25.72,0,0,1,0,55,0,0
1,AAA,2013J,268,383,0.610966,0.428198,6,1,413,1,25.72,0,0,1,0,57,1,0
2,AAA,2013J,268,383,0.610966,0.428198,6,1,413,2,25.72,0,0,1,0,62,2,0
3,AAA,2013J,268,383,0.610966,0.428198,6,1,413,3,25.72,0,0,1,0,51,3,0
4,AAA,2013J,268,383,0.610966,0.428198,6,1,413,4,25.72,0,0,1,0,87,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5495,GGG,2014B,241,833,0.200480,0.216086,10,1,367,245,36.55,0,0,0,1,6,0,0
5496,GGG,2014B,241,833,0.200480,0.216086,10,1,367,246,36.55,0,0,0,1,6,1,0
5497,GGG,2014B,241,833,0.200480,0.216086,10,1,367,247,36.55,0,0,0,1,19,2,0
5498,GGG,2014B,241,833,0.200480,0.216086,10,1,367,248,839.26,1,0,0,1,91,3,0


In [7]:
df = feature_table.copy()

# -----------------------------
# Create a date from day_offset
# -----------------------------
# Choose the dataset start date (change if known)
start_date = pd.Timestamp("2024-01-01")

df["date"] = start_date + pd.to_timedelta(df["day_offset"], unit="D")

# -----------------------------
# Time Features
# -----------------------------
df["day_of_week"] = df["date"].dt.dayofweek
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter

# -----------------------------
# Target column
# -----------------------------
target = "enrollment_num_daily"

# -----------------------------
# Lag Features
# -----------------------------
df["lag_1"] = df[target].shift(1)
df["lag_7"] = df[target].shift(7)
df["lag_14"] = df[target].shift(14)
df["lag_30"] = df[target].shift(30)

# -----------------------------
# Rolling Features
# -----------------------------
df["rolling_mean_7"] = df[target].rolling(7).mean()
df["rolling_mean_30"] = df[target].rolling(30).mean()
df["rolling_std_30"] = df[target].rolling(30).std()

# -----------------------------
# Rename event columns
# -----------------------------
df.rename(columns={
    "marketing_campaign_flag": "campaign_flag",
    "semester_start_flag": "semester_start"
}, inplace=True)

# Remove rows with NaN created by lag/rolling
df = df.dropna().reset_index(drop=True)

# Display engineered features
print(df[[
    "date",
    "day_of_week",
    "week_of_year",
    "month",
    "quarter",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_30",
    "rolling_mean_7",
    "rolling_mean_30",
    "rolling_std_30",
    "holiday_flag",
    "campaign_flag",
    "semester_start"
]].head())

        date  day_of_week  week_of_year  month  quarter  lag_1  lag_7  lag_14  \
0 2024-01-31            2             5      1        1   19.0   26.0    29.0   
1 2024-02-01            3             5      2        1   24.0   35.0    25.0   
2 2024-02-02            4             5      2        1   27.0   38.0    22.0   
3 2024-02-03            5             5      2        1   39.0   28.0    30.0   
4 2024-02-04            6             5      2        1   32.0   17.0    24.0   

   lag_30  rolling_mean_7  rolling_mean_30  rolling_std_30  holiday_flag  \
0    55.0       24.714286        40.166667       22.593993             0   
1    57.0       23.571429        39.166667       22.486906             0   
2    62.0       23.714286        38.400000       22.069795             0   
3    51.0       24.285714        37.766667       21.968133             0   
4    87.0       25.142857        35.633333       20.045609             0   

   campaign_flag  semester_start  
0              0     

In [8]:
df

,code_module,code_presentation,module_presentation_length,historical_student_count,pct_male,pct_higher_ed,total_assignments,is_exam_included,total_learning_materials,day_offset,...,week_of_year,month,quarter,lag_1,lag_7,lag_14,lag_30,rolling_mean_7,rolling_mean_30,rolling_std_30
0,AAA,2013J,268,383,0.610966,0.428198,6,1,413,30,...,5,1,1,19.0,26.0,29.0,55.0,24.714286,40.166667,22.593993
1,AAA,2013J,268,383,0.610966,0.428198,6,1,413,31,...,5,2,1,24.0,35.0,25.0,57.0,23.571429,39.166667,22.486906
2,AAA,2013J,268,383,0.610966,0.428198,6,1,413,32,...,5,2,1,27.0,38.0,22.0,62.0,23.714286,38.400000,22.069795
3,AAA,2013J,268,383,0.610966,0.428198,6,1,413,33,...,5,2,1,39.0,28.0,30.0,51.0,24.285714,37.766667,21.968133
4,AAA,2013J,268,383,0.610966,0.428198,6,1,413,34,...,5,2,1,32.0,17.0,24.0,87.0,25.142857,35.633333,20.045609
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5465,GGG,2014B,241,833,0.200480,0.216086,10,1,367,245,...,36,9,3,1.0,1.0,9.0,10.0,12.857143,14.866667,12.464386
5466,GGG,2014B,241,833,0.200480,0.216086,10,1,367,246,...,36,9,3,6.0,8.0,6.0,11.0,12.571429,14.700000,12.550999
5467,GGG,2014B,241,833,0.200480,0.216086,10,1,367,247,...,36,9,3,6.0,13.0,10.0,12.0,13.428571,14.933333,12.564134
5468,GGG,2014B,241,833,0.200480,0.216086,10,1,367,248,...,36,9,3,19.0,30.0,21.0,9.0,22.142857,17.666667,18.666461


In [9]:
unique_dates = np.sort(df["date"].unique())

print(len(unique_dates))

250


In [10]:
# Features and target
target = "enrollment_num_daily"

# -----------------------------
# Drop objects
# -----------------------------
X = df.drop(columns=[target, "date", "code_module", "code_presentation"])

y = df[target]

# Time-based split (80% train, 20% test)
split = int(len(df) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

In [11]:
import numpy as np

forecast_horizon = 30

unique_dates = np.sort(df["date"].unique())

fold = 1

for cutoff in range(180, len(unique_dates) - forecast_horizon + 1, 30):

    train_dates = unique_dates[:cutoff]
    test_dates = unique_dates[cutoff:cutoff + forecast_horizon]

    train = df[df["date"].isin(train_dates)]
    test = df[df["date"].isin(test_dates)]

    print(f"Fold {fold}")
    print("Train:", train["date"].min(), "->", train["date"].max())
    print("Test :", test["date"].min(), "->", test["date"].max())
    print("Train rows:", len(train))
    print("Test rows :", len(test))
    print()

    fold += 1

Fold 1
Train: 2024-01-01 00:00:00 -> 2024-06-28 00:00:00
Test : 2024-06-29 00:00:00 -> 2024-07-28 00:00:00
Train rows: 3930
Test rows : 660

Fold 2
Train: 2024-01-01 00:00:00 -> 2024-07-28 00:00:00
Test : 2024-07-29 00:00:00 -> 2024-08-27 00:00:00
Train rows: 4590
Test rows : 660



In [ ]:
class TSBacktester:
    """
    A custom time series backtester class for evaluating forecasting models.
    
    Parameters
    ----------
    pred_func : Callable
        A function that makes predictions using historical data.
    start_date : str
        The start date of the backtest period.
    end_date : str
        The end date of the backtest period.
    backtest_freq : str
        The frequency of backtest evaluations (e.g., '1D' for daily progression).
    data_freq : str
        The frequency of the data (e.g., '1D' for daily).
    forecast_horizon : int
        The number of time periods to forecast into the future.
    """

    def __init__(self, pred_func, start_date, end_date, backtest_freq, data_freq, forecast_horizon):
        self.pred_func = pred_func
        self.start_date = start_date
        self.end_date = end_date
        self.backtest_freq = backtest_freq
        self.data_freq = data_freq
        self.forecast_horizon = forecast_horizon
        self.backtest_df = None

    def run_backtest(self, df, target_col, features=None, verbose=False):
        """
        Run the time series backtest using the specified parameters.

        Parameters
        ----------
        df : pd.DataFrame
            The DataFrame containing the time series data.
        target_col : str
            The name of the target column to forecast.
        features : List[str], optional
            List of feature column names. Default is None.
        verbose : bool, optional
            If True, print information about each backtest step. Default is False.
        """

        ts_df = df.copy()

        fcst_dates = pd.date_range(self.start_date, self.end_date, freq=self.backtest_freq)
        backtest_list = []

        for forecast_date in fcst_dates:

            test_ind = pd.date_range(forecast_date, periods=self.forecast_horizon, freq=self.data_freq)

            X_train = ts_df.loc[ts_df.index < forecast_date].copy()
            y_train = X_train.pop(target_col)
            
            X_test = ts_df.loc[ts_df.index.isin(test_ind)].copy()
            y_test = X_test.pop(target_col)

            if verbose:
                print(f"Forecasting as of {forecast_date.date()} ----")
                print(f"Training data: {X_train.index.min().date()} : {X_train.index.max().date()} (n = {len(X_train)})")
                print(f"Test data: {X_test.index.min().date()} : {X_test.index.max().date()} (n = {len(X_test)})")
           
            # get predictions
            y_pred = self.pred_func(X_train, y_train, X_test, self.forecast_horizon, features)
            
            pred_df = pd.DataFrame(
                {
                    "forecast_date": forecast_date,
                    "report_date": y_test.index,
                    "forecast": y_pred,
                    "actual": y_test,
                }
            )

            backtest_list.append(pred_df)


        backtest_df = pd.concat(backtest_list, ignore_index=True)

        self.backtest_df = backtest_df

    def evaluate_backtest(self, metrics):
        """
        Evaluate the backtest using specified performance metrics.

        Parameters
        ----------
        metrics : Dict[str, Callable]
            A dictionary of metric names and their corresponding metric functions.

        Returns
        -------
        Dict[str, Dict[Union[str, int], float]]
            A dictionary containing evaluation scores for each metric.
        """

        if self.backtest_df is None:
            raise ValueError("Backtest was not yet executed! Please run it before evaluating")
        
        backtest_df = self.backtest_df.copy()

        backtest_df["horizon"] = (backtest_df["report_date"] - backtest_df["forecast_date"]).dt.days

        grouped = backtest_df.groupby('horizon')

        scores_dict = {}

        for metric, metric_func in metrics.items():
            
            scores_dict[metric] = {}
            scores_dict[metric]["total"] = metric_func(backtest_df['actual'], backtest_df['forecast'])

            for group, group_df in grouped:
                scores_dict[metric][group] = metric_func(group_df['actual'], group_df['forecast'])

        return scores_dict

In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import numpy as np

naive_pred = y_test.shift(1)

# Fill first prediction
naive_pred.iloc[0] = y_train.iloc[-1]

mae = mean_absolute_error(y_test, naive_pred)
rmse = np.sqrt(mean_squared_error(y_test, naive_pred))
mape = mean_absolute_percentage_error(y_test, naive_pred, sample_weight=y_test)

print("Naive Forecast")
print("MAE :", round(mae,2))
print("RMSE :", round(rmse,2))
print(f"MAPE (fraction): {mape}")
print(f"MAPE (percentage): {mape * 100:.2f}%")

Naive Forecast
MAE : 10.12
RMSE : 18.02
MAPE (fraction): 0.46467003759885905
MAPE (percentage): 46.47%


In [81]:
results = []

target = "enrollment_num_daily"

for fold, cutoff in enumerate(range(180, len(unique_dates)-30+1, 30), start=1):

    train_dates = unique_dates[:cutoff]
    test_dates = unique_dates[cutoff:cutoff+30]

    train = df[df["date"].isin(train_dates)]
    test = df[df["date"].isin(test_dates)]

    X_train = train.drop(columns=["date", target, "code_module", "code_presentation"])
    y_train = train[target]

    X_test = test.drop(columns=["date", target, "code_module", "code_presentation"])
    y_test = test[target]

    naive_pred = y_test.shift(1)

    # Fill first prediction
    naive_pred.iloc[0] = y_train.iloc[-1]

    mae = mean_absolute_error(y_test, naive_pred)
    rmse = np.sqrt(mean_squared_error(y_test, naive_pred))
    mape = mean_absolute_percentage_error(y_test, naive_pred, sample_weight=y_test)

    results.append([fold, mae, rmse, mape])

results = pd.DataFrame(
    results,
    columns=["Fold", "MAE", "RMSE", "MAPE"]
)

print(results)
print("\nAverage")
print(results.mean(numeric_only=True))

   Fold       MAE       RMSE      MAPE
0     1  8.153030  15.259423  0.666812
1     2  8.363636  15.458449  0.703088

Average
Fold     1.500000
MAE      8.258333
RMSE    15.358936
MAPE     0.684950
dtype: float64


In [14]:
!pip install statsmodels

In [54]:
import os
import joblib

# Create checkpoint directory if it doesn't exist
os.makedirs("checkpoints", exist_ok=True)

In [55]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

train = y.iloc[:split]
test = y.iloc[split:]

arima_model = ARIMA(train, order=(5,1,0))
arima_fit = arima_model.fit()

arima_pred = arima_fit.forecast(steps=len(test))

arima_mae = mean_absolute_error(test, arima_pred)
arima_rmse = np.sqrt(mean_squared_error(test, arima_pred))
arima_mape = mean_absolute_percentage_error(test, arima_pred, sample_weight=test)

print("ARIMA")
print("MAE :", arima_mae)
print("RMSE:", arima_rmse)
print(f"MAPE (fraction): {arima_mape}")
print(f"MAPE (percentage): {arima_mape * 100:.2f}%")

ARIMA
MAE : 15.855063207903473
RMSE: 22.82094089414183
MAPE (fraction): 0.7374546805726307
MAPE (percentage): 73.75%


In [56]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_model = SARIMAX(
    train,
    order=(1,1,1),
    seasonal_order=(1,1,1,7)
)

sarima_fit = sarima_model.fit()

sarima_pred = sarima_fit.forecast(len(test))

sarima_mae = mean_absolute_error(test, sarima_pred)
sarima_rmse = np.sqrt(mean_squared_error(test, sarima_pred))
sarima_mape = mean_absolute_percentage_error(test, sarima_pred, sample_weight=test)

print("SARIMA")
print("MAE :", sarima_mae)
print("RMSE:", sarima_rmse)
print(f"MAPE (fraction): {sarima_mape}")
print(f"MAPE (percentage): {sarima_mape * 100:.2f}%")

SARIMA
MAE : 45.8767316395275
RMSE: 52.533963598490146
MAPE (fraction): 2.1354918814410624
MAPE (percentage): 213.55%


In [22]:
!pip install prophet

In [57]:
from prophet import Prophet

prophet_df = df[["date","enrollment_num_daily"]].rename(
    columns={
        "date":"ds",
        "enrollment_num_daily":"y"
    }
)

train = prophet_df.iloc[:split]
test = prophet_df.iloc[split:]

prophet_model = Prophet()

prophet_model.fit(train)

future = prophet_model.make_future_dataframe(periods=len(test))

forecast = prophet_model.predict(future)

prophet_pred = forecast["yhat"].tail(len(test)).values

prophet_mae = mean_absolute_error(test["y"], prophet_pred)
prophet_rmse = np.sqrt(mean_squared_error(test["y"], prophet_pred))
prophet_mape = mean_absolute_percentage_error(test["y"], prophet_pred, sample_weight=test["y"])

print("Prophet")
print("MAE :", prophet_mae)
print("RMSE:", prophet_rmse)
print(f"MAPE (fraction): {prophet_mape}")
print(f"MAPE (percentage): {prophet_mape * 100:.2f}%")

INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


Prophet
MAE : 17.50220900106973
RMSE: 24.24647619816917
MAPE (fraction): 0.8080988005966306
MAPE (percentage): 80.81%


In [34]:
!pip install xgboost

In [62]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    objective="reg:squarederror"
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)

xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_mape = mean_absolute_percentage_error(y_test, xgb_pred, sample_weight=y_test)

print("XGBoost")
print("MAE :", xgb_mae)
print("RMSE:", xgb_rmse)
print(f"MAPE (fraction): {xgb_mape}")
print(f"MAPE (percentage): {xgb_mape * 100:.2f}%")

XGBoost
MAE : 4.279367446899414
RMSE: 5.7683224867827425
MAPE (fraction): 0.1965913325548172
MAPE (percentage): 19.66%


In [84]:
results = []

target = "enrollment_num_daily"

for fold, cutoff in enumerate(range(180, len(unique_dates)-30+1, 30), start=1):

    train_dates = unique_dates[:cutoff]
    test_dates = unique_dates[cutoff:cutoff+30]

    train = df[df["date"].isin(train_dates)]
    test = df[df["date"].isin(test_dates)]

    X_train = train.drop(columns=["date", target, "code_module", "code_presentation"])
    y_train = train[target]

    X_test = test.drop(columns=["date", target, "code_module", "code_presentation"])
    y_test = test[target]

    model = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
        objective="reg:squarederror"
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, naive_pred)
    rmse = np.sqrt(mean_squared_error(y_test, naive_pred))
    mape = mean_absolute_percentage_error(y_test, naive_pred, sample_weight=y_test)

    results.append([fold, mae, rmse, mape])

results = pd.DataFrame(
    results,
    columns=["Fold", "MAE", "RMSE", "MAPE"]
)

print(results)
print("\nAverage")
print(results.mean(numeric_only=True))

   Fold       MAE       RMSE      MAPE
0     1  8.674242  15.365299  0.698143
1     2  8.363636  15.458449  0.703088

Average
Fold     1.500000
MAE      8.518939
RMSE    15.411874
MAPE     0.700616
dtype: float64


In [59]:
!pip install lightgbm

In [61]:
from lightgbm import LGBMRegressor

lgbm_model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    random_state=42
)

lgbm_model.fit(X_train, y_train)

lgbm_pred = lgbm_model.predict(X_test)

lgbm_mae = mean_absolute_error(y_test, lgbm_pred)
lgbm_rmse = np.sqrt(mean_squared_error(y_test, lgbm_pred))
lgbm_mape = mean_absolute_percentage_error(y_test, lgbm_pred, sample_weight=y_test)

print("LightGBM")
print("MAE :", lgbm_mae)
print("RMSE:", lgbm_rmse)
print(f"MAPE (fraction): {lgbm_mape}")
print(f"MAPE (percentage): {lgbm_mape * 100:.2f}%")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000877 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1551
[LightGBM] [Info] Number of data points in the train set: 4376, number of used features: 24
[LightGBM] [Info] Start training from score 15.993601
LightGBM
MAE : 4.2059553192146595
RMSE: 5.530947553145525
MAPE (fraction): 0.19357574596863403
MAPE (percentage): 19.36%


In [85]:
results = []

target = "enrollment_num_daily"

for fold, cutoff in enumerate(range(180, len(unique_dates)-30+1, 30), start=1):

    train_dates = unique_dates[:cutoff]
    test_dates = unique_dates[cutoff:cutoff+30]

    train = df[df["date"].isin(train_dates)]
    test = df[df["date"].isin(test_dates)]

    X_train = train.drop(columns=["date", target, "code_module", "code_presentation"])
    y_train = train[target]

    X_test = test.drop(columns=["date", target, "code_module", "code_presentation"])
    y_test = test[target]

    model = LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        random_state=42
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, naive_pred)
    rmse = np.sqrt(mean_squared_error(y_test, naive_pred))
    mape = mean_absolute_percentage_error(y_test, naive_pred, sample_weight=y_test)

    results.append([fold, mae, rmse, mape])

results = pd.DataFrame(
    results,
    columns=["Fold", "MAE", "RMSE", "MAPE"]
)

print(results)
print("\nAverage")
print(results.mean(numeric_only=True))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000929 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1494
[LightGBM] [Info] Number of data points in the train set: 3930, number of used features: 23
[LightGBM] [Info] Start training from score 19.386260
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000566 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1538
[LightGBM] [Info] Number of data points in the train set: 4590, number of used features: 23
[LightGBM] [Info] Start training from score 18.100654
   Fold       MAE       RMSE      MAPE
0     1  8.674242  15.365299  0.698143
1     2  8.363636  15.458449  0.703088

Average
Fold     1.500000
MAE      8.518939
RMSE    15.411874
MAPE    

In [13]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

target = "enrollment_num_daily"

# Features
X = df.drop(columns=[target], errors="ignore")
X = X.drop(columns=["date"], errors="ignore")
X = X.drop(columns=["code_module"], errors="ignore")
X = X.drop(columns=["code_presentation"], errors="ignore")

# Encode categorical columns
X = pd.get_dummies(X, drop_first=True).astype(float)

y = df[target].values

# Scale
X_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X_scaled = X_scaler.fit_transform(X)
y_scaled = y_scaler.fit_transform(y.reshape(-1, 1))

# Create sequences
look_back = 30

X_seq = []
y_seq = []

for i in range(look_back, len(X_scaled)):
    X_seq.append(X_scaled[i-look_back:i])
    y_seq.append(y_scaled[i])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

# Train/test split
split = int(len(X_seq) * 0.8)

X_train = X_seq[:split]
X_test = X_seq[split:]

y_train = y_seq[:split]
y_test = y_seq[split:]

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

X_train shape: (4352, 30, 25)
X_test shape : (1088, 30, 25)


In [14]:
import os

CHECKPOINT_DIR = "checkpoints"
checkpoint = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [15]:
from tensorflow.keras.callbacks import ModelCheckpoint
import os

os.makedirs("checkpoints", exist_ok=True)

checkpoint = ModelCheckpoint(
    filepath="checkpoints/lstm.keras",
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False,
    mode="min",
    verbose=1
)

In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)
import numpy as np

# Build model
lstm_model = Sequential([
    LSTM(64, return_sequences=True,
         input_shape=(look_back, X_train.shape[2])),
    Dropout(0.2),
    LSTM(32),
    Dense(16, activation="relu"),
    Dense(1)
])

lstm_model.compile(
    optimizer="adam",
    loss="mse"
)

# Train
history = lstm_model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    callbacks=[checkpoint],
    verbose=1
)

# Predict
lstm_pred = lstm_model.predict(X_test, verbose=0)

# Convert back to original scale
y_test_inv = y_scaler.inverse_transform(y_test)
lstm_pred_inv = y_scaler.inverse_transform(lstm_pred)

# Evaluation
lstm_mae = mean_absolute_error(y_test_inv, lstm_pred_inv)
lstm_rmse = np.sqrt(mean_squared_error(y_test_inv, lstm_pred_inv))
lstm_mape = mean_absolute_percentage_error(y_test_inv, lstm_pred_inv) * 100
lstm_wape = (
    np.sum(np.abs(y_test_inv - lstm_pred_inv))
    / np.sum(np.abs(y_test_inv))
) * 100

print("LSTM Results")
print(f"MAE : {lstm_mae:.4f}")
print(f"RMSE: {lstm_rmse:.4f}")
print(f"MAPE: {lstm_mape:.2f}%")
print(f"WAPE: {lstm_wape:.2f}%")

Epoch 1/20
121/123 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 0.0075
Epoch 1: val_loss improved from None to 0.00658, saving model to checkpoints/lstm.keras

Epoch 1: finished saving model to checkpoints/lstm.keras
123/123 ━━━━━━━━━━━━━━━━━━━━ 11s 43ms/step - loss: 0.0069 - val_loss: 0.0066
Epoch 2/20
122/123 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 0.0059
Epoch 2: val_loss did not improve from 0.00658
123/123 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - loss: 0.0057 - val_loss: 0.0080
Epoch 3/20
123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.0057
Epoch 3: val_loss improved from 0.00658 to 0.00539, saving model to checkpoints/lstm.keras

Epoch 3: finished saving model to checkpoints/lstm.keras
123/123 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - loss: 0.0051 - val_loss: 0.0054
Epoch 4/20
122/123 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - loss: 0.0046
Epoch 4: val_loss improved from 0.00539 to 0.00476, saving model to checkpoints/lstm.keras

Epoch 4: finished saving model to checkpoints/lstm.keras
123/123 

In [17]:
# Evaluation
lstm_mae = mean_absolute_error(y_test_inv, lstm_pred_inv)
lstm_rmse = np.sqrt(mean_squared_error(y_test_inv, lstm_pred_inv))
lstm_mape = mean_absolute_percentage_error(y_test_inv, lstm_pred_inv, sample_weight=y_test_inv) * 100
lstm_wape = (
    np.sum(np.abs(y_test_inv - lstm_pred_inv))
    / np.sum(np.abs(y_test_inv))
) * 100

print("LSTM Results")
print(f"MAE : {lstm_mae:.4f}")
print(f"RMSE: {lstm_rmse:.4f}")
print(f"MAPE: {lstm_mape:.2f}%")
print(f"WAPE: {lstm_wape:.2f}%")

LSTM Results
MAE : 6.3830
RMSE: 12.7204
MAPE: 28.91%
WAPE: 30.10%


In [89]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

look_back = 30
forecast_horizon = 30
target = "enrollment_num_daily"

results = []

for fold, cutoff in enumerate(
    range(180, len(unique_dates) - forecast_horizon + 1, forecast_horizon),
    start=1
):

    # Split by dates
    train_dates = unique_dates[:cutoff]
    test_dates = unique_dates[cutoff:cutoff + forecast_horizon]

    train = df[df["date"].isin(train_dates)]
    test = df[df["date"].isin(test_dates)]

    X_train = train.drop(columns=["date", target, "code_module", "code_presentation"])
    y_train = train[target]

    X_test = test.drop(columns=["date", target, "code_module", "code_presentation"])
    y_test = test[target]

    # Scale features
    scaler = MinMaxScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Create sequences
    X_train_seq, y_train_seq = [], []
    for i in range(len(X_train) - look_back):
        X_train_seq.append(X_train[i:i+look_back])
        y_train_seq.append(y_train.iloc[i+look_back])

    X_test_seq, y_test_seq = [], []
    for i in range(len(X_test) - look_back):
        X_test_seq.append(X_test[i:i+look_back])
        y_test_seq.append(y_test.iloc[i+look_back])

    X_train_seq = np.array(X_train_seq)
    y_train_seq = np.array(y_train_seq)

    X_test_seq = np.array(X_test_seq)
    y_test_seq = np.array(y_test_seq)

    # Build model
    model = Sequential([
        LSTM(64, return_sequences=True,
             input_shape=(look_back, X_train_seq.shape[2])),
        Dropout(0.2),
        LSTM(32),
        Dense(16, activation="relu"),
        Dense(1)
    ])

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    model.fit(
        X_train_seq,
        y_train_seq,
        epochs=20,
        batch_size=32,
        validation_split=0.1,
        verbose=0
    )

    pred = model.predict(X_test_seq, verbose=0).flatten()

    mae = mean_absolute_error(y_test_seq, pred)
    rmse = np.sqrt(mean_squared_error(y_test_seq, pred))
    mape = mean_absolute_percentage_error(y_test_seq, pred) * 100
    wape = np.sum(np.abs(y_test_seq - pred)) / np.sum(np.abs(y_test_seq)) * 100

    results.append([
        fold,
        mae,
        rmse,
        mape,
        wape
    ])

results = pd.DataFrame(
    results,
    columns=["Fold", "MAE", "RMSE", "MAPE (%)", "WAPE (%)"]
)

print(results)
print("\nAverage")
print(results.mean(numeric_only=True))

   Fold       MAE       RMSE      MAPE (%)   WAPE (%)
0     1  8.446436  12.024194  8.775934e+17  83.470661
1     2  7.072378  11.653302  7.166944e+17  65.475360

Average
Fold        1.500000e+00
MAE         7.759407e+00
RMSE        1.183875e+01
MAPE (%)    7.971439e+17
WAPE (%)    7.447301e+01
dtype: float64


In [69]:
!ls '/content/checkpoints'

lstm.keras


In [159]:
import os
import joblib
from prophet.serialize import model_to_json

CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ARIMA
joblib.dump(arima_fit, f"{CHECKPOINT_DIR}/arima.pkl")

# SARIMA
joblib.dump(sarima_fit, f"{CHECKPOINT_DIR}/sarima.pkl")

# Prophet
with open(f"{CHECKPOINT_DIR}/prophet.json", "w") as f:
    f.write(model_to_json(prophet_model))

# XGBoost
xgb_model.save_model(f"{CHECKPOINT_DIR}/xgboost.json")

# LightGBM
lgbm_model.booster_.save_model(f"{CHECKPOINT_DIR}/lightgbm.txt")

# LSTM
# lstm_model.save(f"{CHECKPOINT_DIR}/lstm.keras")

print("All models saved successfully.")

All models saved successfully.


In [166]:
import os
import joblib
import pickle


CHECKPOINT_PKL = "checkpoints_pkl"
os.makedirs(CHECKPOINT_PKL, exist_ok=True)
joblib.dump(arima_fit, f"{CHECKPOINT_PKL}/arima.pkl")
joblib.dump(sarima_fit, f"{CHECKPOINT_PKL}/sarima.pkl")
with open(f"{CHECKPOINT_PKL}/prophet.pkl", "wb") as f:
    pickle.dump(prophet_model, f)
joblib.dump(xgb_model, f"{CHECKPOINT_PKL}/xgboost.pkl")
joblib.dump(lgbm_model, f"{CHECKPOINT_PKL}/lightgbm.pkl")

print("All available models have been saved to the 'checkpoints' folder.")

All available models have been saved to the 'checkpoints' folder.


In [169]:
!ls "/content/drive/MyDrive/EdTech Project/Code/checkpoints_pkl"

arima.pkl  lightgbm.pkl  prophet.pkl  sarima.pkl  xgboost.pkl


In [168]:
!cp -r checkpoints_pkl "/content/drive/MyDrive/EdTech Project/Code/checkpoints_pkl"
